# 🐟 FishFresh — YOLO Fish Detector → TFLite (FINAL)

This notebook loads a **pretrained YOLOv8 (OpenImages‑V7)** model, filters outputs to **Fish** only, predicts on your **tray photos** under `assets/samples/`, and exports **both** FP16 and FP32 **TFLite** files for your Flutter app. It also includes a **test/evaluation from assets** with optional labels and writes a **per‑image CSV** summary.

**Flow:** Detect (Fish only) → crop (in app) → classify with your MobileNet (species + freshness).

## 0) Environment check
If you haven't installed packages in this environment yet, run the line below once (then restart kernel):
```python
# %pip install --prefer-binary ultralytics onnxruntime tensorflow-cpu==2.16.1
```


In [2]:
import sys, numpy as np, onnx, onnxruntime, ultralytics
print("python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("onnx:", onnx.__version__)
print("onnxruntime:", onnxruntime.__version__)
print("ultralytics:", ultralytics.__version__)
try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except Exception as e:
    print("tensorflow: import failed ->", repr(e))


python: 3.11.9
numpy: 1.26.4
onnx: 1.18.0
onnxruntime: 1.23.2
ultralytics: 8.3.228
tensorflow: 2.15.0


## 1) Paths & folders
Put a few tray images (jpg/png) in **`assets/samples/`** to test.

In [3]:
from pathlib import Path

ROOT = Path.cwd()

# All assets inside model/
MODEL_DIR = ROOT / ''

ASSETS  = MODEL_DIR / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)

SAMPLES = ASSETS / 'samples'
SAMPLES.mkdir(parents=True, exist_ok=True)

# Output folder also inside model/
OUT = MODEL_DIR / 'runs_fishdet'
OUT.mkdir(parents=True, exist_ok=True)


print("ROOT    =", ROOT)
print("MODEL   =", MODEL_DIR)
print("ASSETS  =", ASSETS)
print("SAMPLES =", SAMPLES, "(put tray photos here)")
print("OUT     =", OUT)


ROOT    = C:\aziztebbeng\2025-CP_Fishfresh\model
MODEL   = C:\aziztebbeng\2025-CP_Fishfresh\model
ASSETS  = C:\aziztebbeng\2025-CP_Fishfresh\model\assets
SAMPLES = C:\aziztebbeng\2025-CP_Fishfresh\model\assets\samples (put tray photos here)
OUT     = C:\aziztebbeng\2025-CP_Fishfresh\model\runs_fishdet


## 2) Load YOLOv8 (OpenImages‑V7) and resolve `fish_id`
- `yolov8n-oiv7.pt` = fastest/smallest
- `yolov8s-oiv7.pt` = a bit more accurate (slower)

In [4]:
from ultralytics import YOLO

MODEL_NAME = 'yolov8s-oiv7.pt'  # or 'yolov8s-oiv7.pt'
model = YOLO(MODEL_NAME)        # auto-downloads on first use
print('[✓] Loaded YOLO model:', MODEL_NAME)

# Resolve the class id for 'Fish' (case-insensitive)
fish_id = next((int(k) for k, v in model.names.items() if str(v).lower() == 'fish'), None)
assert fish_id is not None, "Fish class not found in this checkpoint."
print('Fish class id =', fish_id)

[✓] Loaded YOLO model: yolov8s-oiv7.pt
Fish class id = 192


## 3) Predict on assets/samples (Fish only)
Tune `IMGSZ`, `CONF`, `MAXDET` as needed. Annotated results saved under `runs_fishdet/pred_trays/`.

In [5]:
IMGSZ  = 768          # keep same as training size
CONF   = 0.18         # 0.15–0.25 is usually good for trays
IOU    = 0.45         # less aggressive NMS than 0.60
MAXDET = 400          # enough for a tray

img_list = [*SAMPLES.glob('*.jpg'),
            *SAMPLES.glob('*.jpeg'),
            *SAMPLES.glob('*.png')]
print('Found', len(img_list), 'images in', SAMPLES)

if img_list:
    results = model.predict(
        source=str(SAMPLES),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAXDET,
        classes=[fish_id],   # or None if you want all classes
        save=True,           # turn ON so you can inspect the tray outputs
        save_txt=False,
        project=str(OUT),
        name='pred_trays',
        exist_ok=True,
        verbose=False,
        workers=0,
        # device=0
    )
    print('[√] Done. See annotated results in:', OUT/'pred_trays')
else:
    print('[!] No images found in', SAMPLES)


Found 0 images in C:\aziztebbeng\2025-CP_Fishfresh\model\assets\samples
[!] No images found in C:\aziztebbeng\2025-CP_Fishfresh\model\assets\samples


## 4) Export **both** TFLite files + config
Writes to `runs_fishdet/export/`:
- `fishdet_fp16.tflite`
- `fishdet_fp32.tflite`
- `detector_config.json`

In [6]:
from pathlib import Path
import shutil
from datetime import datetime
import json

# === PATHS ===
ROOT = Path.cwd()                              # ...\2025-CP_Fishfresh\model
PROJECT_ROOT = ROOT.parent                     # ...\2025-CP_Fishfresh

# THIS IS YOUR FLUTTER ASSETS FOLDER:
ASSETS_MODEL_DIR = PROJECT_ROOT / "assets" / "model"   # C:\aziztebbeng\2025-CP_Fishfresh\assets\model
ASSETS_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# === SETTINGS ===
IMGSZ = int(globals().get("IMGSZ", 768))    # must match your YOLO input size
MAXDET = int(globals().get("MAXDET", 400))
DISPLAY_CONF = float(globals().get("DISPLAY_CONF", 0.35))

# === TIMESTAMP ===
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
human_time = datetime.now().strftime("%A, %B %d, %Y at %I:%M:%S %p")

print("========================================")
print(" YOLOv8 → TFLite export for FishFresh")
print(" Weights     :", MODEL_NAME)
print(" Image size  :", IMGSZ)
print(" Export time :", human_time)
print(" Assets dir  :", ASSETS_MODEL_DIR)
print("========================================")

from ultralytics import YOLO

# model should already be loaded earlier as `model`
# If not, uncomment:
# model = YOLO(MODEL_NAME)

# 1) Export YOLO model directly to TFLite (float32)
tflite_path_raw = Path(
    model.export(
        format="tflite",
        imgsz=IMGSZ,
        int8=False,   # CPU float32
        half=False,   # no FP16
        dynamic=False,
    )
)
print("[✓] Raw TFLite exported by Ultralytics:", tflite_path_raw)

# 2) Copy & rename into Flutter assets/model/
final_tflite_name = f"fishdet_fp32_{timestamp}.tflite"
final_tflite_path = ASSETS_MODEL_DIR / final_tflite_name
shutil.copy2(tflite_path_raw, final_tflite_path)
print("[✓] Copied to Flutter assets:", final_tflite_path)

# 3) Build fishdet_config.json next to it
detector_config = {
    "detector_name": f"fishdet_yolov8s_oiv7_{IMGSZ}",
    "input_size": IMGSZ,
    "conf_threshold": DISPLAY_CONF,
    "iou_threshold": 0.60,
    "max_detections": MAXDET,
    # IMPORTANT: this is the path Flutter will use
    "tflite_asset": f"assets/model/{final_tflite_name}",
    "label_map": {
        "0": "fish"
    },
    "export_info": {
        "tflite_file": final_tflite_name,
        "export_time": human_time
    }
}

json_out = ASSETS_MODEL_DIR / "fishdet_config.json"
with open(json_out, "w") as f:
    json.dump(detector_config, f, indent=2)

print("[✓] fishdet_config.json exported to:", json_out)
print("✅ Export finished at:", human_time)


 YOLOv8 → TFLite export for FishFresh
 Weights     : yolov8s-oiv7.pt
 Image size  : 768
 Export time : Sunday, December 07, 2025 at 02:44:18 AM
 Assets dir  : C:\aziztebbeng\2025-CP_Fishfresh\assets\model
Ultralytics 8.3.228  Python-3.11.9 torch-2.9.0+cpu CPU (12th Gen Intel Core i3-1215U)
YOLOv8s summary (fused): 72 layers, 11,358,171 parameters, 0 gradients, 29.7 GFLOPs

PyTorch: starting from 'yolov8s-oiv7.pt' with input shape (1, 3, 768, 768) BCHW and output shape(s) (1, 605, 12096) (21.9 MB)
requirements: Ultralytics requirements ['ai-edge-litert>=1.2.0', 'protobuf>=5'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command 'pip install --no-cache-dir "ai-edge-litert>=1.2.0" "protobuf>=5" --extra-index-url https://pypi.ngc.nvidia.com' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command 'pip install --no-cache-dir "ai-edge-litert>=1.2.0" "protobuf>=5" --extra-index-url https://pypi.ngc.nvidia.com' returned non-zero exit status 1.
WARNING requirements: 

## 6) Evaluate directly from **assets/**
Two modes:
- **With labels** at `assets/labels/*.txt` (YOLO format; class `0 = fish`) → reports **mAP@0.5**, **Precision/Recall/F1** at your chosen `CONF_EVAL`, and writes a **per‑image CSV**.
- **Without labels** → reports **Detection Rate**, **avg detections/image**, **avg confidence** and writes a per‑image CSV with detections only.

> You can adjust `CONF_EVAL` below (default **0.35**).

In [10]:
# === Evaluate on assets/ — mAP@0.5 + best-F1 & summary ===
from pathlib import Path
from PIL import Image
import numpy as np, csv

# --- paths ---
IMG_DIR   = SAMPLES                  # model/assets/samples
LBL_DIR   = ASSETS / "labels"        # model/assets/labels
CSV_PATH  = ASSETS_MODEL_DIR / "assets_eval_summary.csv"

DISPLAY_CONF = float(globals().get("DISPLAY_CONF", 0.02))  # what you show to adviser
IOU_MATCH    = 0.45                    # IoU for TP/FP
IMGSZ_EVAL   = IMGSZ
MAXDET_EVAL  = MAXDET

# make sure model exists
try:
    model  # noqa
except NameError:
    raise RuntimeError("Run the cell that loads YOLO (MODEL_NAME / model = YOLO(...)) first.")

print("[*] IMG_DIR:", IMG_DIR)
print("[*] LBL_DIR:", LBL_DIR)

# ========== helpers ==========
def box_iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter   = inter_w * inter_h
    if inter <= 0:
        return 0.0
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-9)

# --- read GT boxes, using labels as source-of-truth ---
all_gts   = {}       # key = image filename, value = array [N,4]
img_paths = []       # list of Path objects for images that *have* labels
total_gt  = 0

label_files = sorted(LBL_DIR.glob("*.txt"))
print(f"[*] Found {len(label_files)} label files")

for lbl in label_files:
    stem = lbl.stem   # e.g. "1_06_21-B1.jpg.rf.xxxxx"

    # Try to find matching image in samples folder
    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        cand = IMG_DIR / f"{stem}{ext}"
        if cand.exists():
            img_path = cand
            break

    if img_path is None:
        # no matching image in samples for this label -> skip
        continue

    # open image to get width/height
    w, h = Image.open(img_path).size
    boxes = []
    with open(lbl, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            # YOLO bbox format: class cx cy w h  (5 numbers)
            # YOLO seg format:  class x1 y1 x2 y2 ... xN yN  (>= 6 numbers)
            nums = list(map(float, parts[1:]))

            if len(nums) == 4:
                # normal bbox
                cx, cy, bw, bh = nums
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
            elif len(nums) >= 6:
                # polygon → convert to bbox
                xs = nums[0::2]
                ys = nums[1::2]
                x1 = min(xs) * w
                y1 = min(ys) * h
                x2 = max(xs) * w
                y2 = max(ys) * h
            else:
                continue

            boxes.append([x1, y1, x2, y2])


    boxes = np.array(boxes, float) if boxes else np.zeros((0, 4), float)
    all_gts[img_path.name] = boxes
    img_paths.append(img_path)
    total_gt += len(boxes)

print(f"[*] Images with labels: {len(img_paths)}")
print(f"[*] Total GT boxes    : {total_gt}")

if not img_paths:
    print("[!] No matching images+labels found in assets/samples & assets/labels.")
    raise SystemExit

if total_gt == 0:
    print("[!] Label files found but they contain no usable YOLO boxes (0  cx cy w h).")
    raise SystemExit

# --- run detector on labeled images in small batches (avoids MemoryError) ---
BATCH_EVAL = 16   # you can make this 8 if RAM is still a problem

all_preds = []   # for full PR curve
rows      = []   # for CSV summary at DISPLAY_CONF

for start in range(0, len(img_paths), BATCH_EVAL):
    batch_paths = img_paths[start:start + BATCH_EVAL]
    print(f"[*] Evaluating batch {start}–{start + len(batch_paths) - 1} / {len(img_paths)-1}")
    
    batch_results = model.predict(
        source=[str(p) for p in batch_paths],
        imgsz=IMGSZ_EVAL,
        conf=0.001,
        iou=0.7,
        max_det=MAXDET_EVAL,
        verbose=False,
    )

    for img_path, r in zip(batch_paths, batch_results):
        boxes = r.boxes
        xyxy  = boxes.xyxy.cpu().numpy() if boxes is not None else np.zeros((0, 4))
        confs = boxes.conf.cpu().numpy() if boxes is not None else np.zeros((0,))

        # --- summary at chosen DISPLAY_CONF ---
        keep        = confs >= DISPLAY_CONF
        xyxy_disp   = xyxy[keep]
        confs_disp  = confs[keep]
        gt_boxes    = all_gts[img_path.name].copy()
        used        = np.zeros(len(gt_boxes), bool)
        tp_disp     = 0
        fp_disp     = 0

        for b in xyxy_disp:
            ious = np.array([box_iou_xyxy(b, g) for g in gt_boxes]) if len(gt_boxes) else np.zeros(0)
            if len(ious) and ious.max() >= IOU_MATCH:
                j = ious.argmax()
                if not used[j]:
                    tp_disp += 1
                    used[j] = True
                else:
                    fp_disp += 1
            else:
                fp_disp += 1

        rows.append([img_path.name, len(gt_boxes), len(xyxy_disp), tp_disp, fp_disp])

        # --- store all preds for PR/mAP ---
        for b, s in zip(xyxy, confs):
            all_preds.append({"image": img_path.name, "score": float(s), "box": b})


# --- build precision–recall curve ---
all_preds = sorted(all_preds, key=lambda d: d["score"], reverse=True)
gt_used   = {name: np.zeros(len(g), bool) for name, g in all_gts.items()}

tps = fps = 0
precisions = []
recalls    = []

for pred in all_preds:
    img_name = pred["image"]
    box      = pred["box"]
    gts      = all_gts[img_name]
    used     = gt_used[img_name]

    if len(gts) == 0:
        fps += 1
    else:
        ious = np.array([box_iou_xyxy(box, g) for g in gts])
        j    = ious.argmax()
        if ious[j] >= IOU_MATCH and not used[j]:
            tps += 1
            used[j] = True
        else:
            fps += 1

    precisions.append(tps / (tps + fps + 1e-9))
    recalls.append(tps / total_gt)

# --- compute AP (area under PR) ---
precisions = np.array(precisions)
recalls    = np.array(recalls)

order = np.argsort(recalls)
r = recalls[order]
p = precisions[order]

mrec = np.concatenate(([0.0], r, [1.0]))
mpre = np.concatenate(([0.0], p, [0.0]))
for i in range(len(mpre)-1, 0, -1):
    mpre[i-1] = max(mpre[i-1], mpre[i])
ap50 = np.sum((mrec[1:] - mrec[:-1]) * mpre[1:])

# --- best F1 over all thresholds ---
F1        = 2 * p * r / (p + r + 1e-9)
best_idx  = np.argmax(F1)
best_P    = p[best_idx]
best_R    = r[best_idx]
best_F1   = F1[best_idx]
BEST_CONF = all_preds[best_idx]["score"]

# --- metrics at DISPLAY_CONF (using rows) ---
tp_disp_total = sum(row[3] for row in rows)
fp_disp_total = sum(row[4] for row in rows)
P_disp = tp_disp_total / (tp_disp_total + fp_disp_total + 1e-9)
R_disp = tp_disp_total / total_gt
F1_disp = 2 * P_disp * R_disp / (P_disp + R_disp + 1e-9)

print("=== Labeled evaluation (YOLO TXT in assets/labels) ===")
print(f"Images          : {len(img_paths)}")
print(f"GT fish boxes   : {total_gt}")
print(f"mAP@0.5         : {ap50*100:.2f}%")
print(f"Best F1 at conf={BEST_CONF:.2f} -> P={best_P*100:.2f}%  R={best_R*100:.2f}%  F1={best_F1*100:.2f}%")
print(f"At conf={DISPLAY_CONF:.2f}     -> P={P_disp*100:.2f}%  R={R_disp*100:.2f}%  F1={F1_disp*100:.2f}%")

with open(CSV_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image","gt_fish","pred_fish_at_conf","TP_at_conf","FP_at_conf"])
    w.writerows(rows)
print("[✓] Wrote CSV:", CSV_PATH)


[*] IMG_DIR: C:\aziztebbeng\2025-CP_Fishfresh\model\assets\samples
[*] LBL_DIR: C:\aziztebbeng\2025-CP_Fishfresh\model\assets\labels
[*] Found 235 label files
[*] Images with labels: 235
[*] Total GT boxes    : 1287
[*] Evaluating batch 0–15 / 234
[*] Evaluating batch 16–31 / 234
[*] Evaluating batch 32–47 / 234
[*] Evaluating batch 48–63 / 234
[*] Evaluating batch 64–79 / 234
[*] Evaluating batch 80–95 / 234
[*] Evaluating batch 96–111 / 234
[*] Evaluating batch 112–127 / 234
[*] Evaluating batch 128–143 / 234
[*] Evaluating batch 144–159 / 234
[*] Evaluating batch 160–175 / 234
[*] Evaluating batch 176–191 / 234
[*] Evaluating batch 192–207 / 234
[*] Evaluating batch 208–223 / 234
[*] Evaluating batch 224–234 / 234
=== Labeled evaluation (YOLO TXT in assets/labels) ===
Images          : 235
GT fish boxes   : 1287
mAP@0.5         : 89.60%
Best F1 at conf=0.32 -> P=90.03%  R=82.05%  F1=85.85%
At conf=0.35     -> P=91.33%  R=80.19%  F1=85.40%
[✓] Wrote CSV: C:\aziztebbeng\2025-CP_Fishfr

In [11]:
# === YOLO Fish Detector — Model Verification Summary ===
from ultralytics import YOLO
import torch, ultralytics
from pathlib import Path

# --- edit this line if you want a nicer description in the printout ---
PRETRAINED_SOURCE = "YOLOv8-small fish segmentation model (OpenImages-V7, public pretrained weights)"

print(f"PyTorch version : {torch.__version__}")
print(f"Ultralytics ver.: {ultralytics.__version__}")

# --- get or load model ---
try:
    yolo = model              # reuse existing model from notebook
    loaded_from = getattr(yolo, "ckpt_path", None)
except NameError:
    ckpt = Path("runs_fishdet/train/weights/best.pt")
    yolo = YOLO(str(ckpt))
    loaded_from = ckpt

print("\nModel source metadata")
print("---------------------")
print(f"Model wrapper class : {type(yolo).__name__}")
print(f"Underlying module   : {type(yolo.model).__name__}")
print(f"Task                : {yolo.task}")
print(f"Weights loaded from : {loaded_from}")

# --- class info ---
names = getattr(yolo, "names", None) or getattr(yolo.model, "names", {})
num_classes = len(names)


# --- parameter counts ---
total_params = sum(p.numel() for p in yolo.model.parameters())
trainable_params = sum(p.numel() for p in yolo.model.parameters() if p.requires_grad)

print("\nParameter statistics")
print("--------------------")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {total_params - trainable_params:,}")

# --- architecture config (depth/width, etc. if available) ---
yaml_cfg = getattr(yolo.model, "yaml", None)
if isinstance(yaml_cfg, dict):
    depth_mult = yaml_cfg.get("depth_multiple", None)
    width_mult = yaml_cfg.get("width_multiple", None)
else:
    depth_mult = width_mult = None

print("\nArchitecture configuration")
print("--------------------------")
print("Base architecture   : YOLOv8-small (segmentation head)")
if depth_mult is not None:
    print(f"Depth multiplier    : {depth_mult}")
if width_mult is not None:
    print(f"Width multiplier    : {width_mult}")

# --- runtime config from your notebook globals ---
imgsz_used   = int(globals().get("IMGSZ", 768))
maxdet_used  = int(globals().get("MAXDET", 400))
conf_default = float(globals().get("CONF", 0.35))
iou_match    = float(globals().get("IOU_MATCH", 0.50))

print("\nNotebook runtime configuration")
print("------------------------------")
print(f"Input size (inference) : {imgsz_used} x {imgsz_used} (RGB)")
print(f"Max detections / image : {maxdet_used}")
print(f"Default conf threshold : {conf_default}")
print(f"IoU threshold (metrics): {iou_match}")

print("\nPretraining information")
print("-----------------------")
print("Backbone             : YOLOv8-small segmentation backbone")
print(f"Pretrained on        : {PRETRAINED_SOURCE}")

print("\n✅ YOLO Fish Detector model verification summary complete.")


PyTorch version : 2.9.0+cpu
Ultralytics ver.: 8.3.228

Model source metadata
---------------------
Model wrapper class : YOLO
Underlying module   : DetectionModel
Task                : detect
Weights loaded from : yolov8s-oiv7.pt

Parameter statistics
--------------------
Total parameters     : 11,358,171
Trainable parameters : 0
Frozen parameters    : 11,358,171

Architecture configuration
--------------------------
Base architecture   : YOLOv8-small (segmentation head)

Notebook runtime configuration
------------------------------
Input size (inference) : 768 x 768 (RGB)
Max detections / image : 400
Default conf threshold : 0.18
IoU threshold (metrics): 0.45

Pretraining information
-----------------------
Backbone             : YOLOv8-small segmentation backbone
Pretrained on        : YOLOv8-small fish segmentation model (OpenImages-V7, public pretrained weights)

✅ YOLO Fish Detector model verification summary complete.
